# Dark Chat - Fine-tuning Qwen2.5 1.5B

In [ ]:
!pip install -q transformers datasets trl peft accelerate

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print("GPU:", torch.cuda.get_device_name(0))

training_data = [
    {"instruction": "Привет", "output": "Привет! Чем могу помочь?"},
    {"instruction": "Кто ты?", "output": "Я Dark Chat — AI-ассистент. Отвечаю на вопросы, пишу код."},
    {"instruction": "Что ты умеешь?", "output": "Отвечаю на вопросы, пишу код, объясняю вещи простыми словами."},
    {"instruction": "Напиши hello world на Python", "output": "print('Hello, World!')"},
    {"instruction": "Напиши hello world на JavaScript", "output": "console.log('Hello, World!');"},
    {"instruction": "Напиши hello world на Go", "output": "fmt.Println('Hello, World!')"},
    {"instruction": "Что такое переменная?", "output": "Переменная — именованная область памяти для хранения значения."},
    {"instruction": "Объясни интернет", "output": "Интернет — глобальная сеть компьютеров для обмена данными."},
    {"instruction": "2+2=", "output": "4"},
    {"instruction": "Что такое AI?", "output": "AI — искусственный интеллект. Программы, которые учатся и принимают решения."},
    {"instruction": "Факториал на Python", "output": "def factorial(n):\n    if n<=1: return 1\n    return n*factorial(n-1)"},
    {"instruction": "Что такое массив?", "output": "Массив — структура данных со списком элементов. Доступ по индексу."},
    {"instruction": "Расскажи анекдот", "output": "Программист решает проблему, о которой ты не знал, способом который не понимаешь."},
    {"instruction": "Как создать сайт?", "output": "HTML + CSS + JavaScript. Фреймворки: React, Vue, Django."},
    {"instruction": "Что такое API?", "output": "API — интерфейс общения программ друг с другом."},
    {"instruction": "Как работает Python?", "output": "Интерпретируемый язык. Код выполняется CPython построчно."},
    {"instruction": "Что такое БД?", "output": "База данных — хранилище информации. SQLite, PostgreSQL, MySQL."},
    {"instruction": "Сортировка пузырьком", "output": "for i in range(n):\n    for j in range(0,n-i-1):\n        if a[j]>a[j+1]: a[j],a[j+1]=a[j+1],a[j]"},
    {"instruction": "Что такое Git?", "output": "Git — система контроля версий для отслеживания изменений кода."},
    {"instruction": "Объясни Docker", "output": "Docker — контейнеризация. Упаковывает приложение с зависимостями."},
    {"instruction": "Что такое REST?", "output": "REST API — стиль API с методами GET, POST, PUT, DELETE."},
    {"instruction": "Чтение файла Python", "output": "with open('file.txt') as f:\n    content = f.read()"},
    {"instruction": "Что такое рекурсия?", "output": "Рекурсия — функция вызывает сама себя. Нужен выход."},
    {"instruction": "API на Python", "output": "from fastapi import FastAPI\napp = FastAPI()"},
    {"instruction": "Что такое SQL?", "output": "SQL — язык запросов: SELECT, INSERT, UPDATE, DELETE."},
    {"instruction": "Чат-бот Python", "output": "while True:\n    msg=input('Ты: ')\n    if msg=='пока': break"},
    {"instruction": "Что такое ООП?", "output": "ООП — объектно-ориентированное программирование."},
    {"instruction": "Async/await", "output": "Асинхронное программирование. Выполняет задачи параллельно."},
    {"instruction": "Middleware", "output": "Промежуточный слой обработки запросов и ответов."},
    {"instruction": "HTTP-сервер", "output": "from http.server import HTTPServer, SimpleHTTPRequestHandler"},
    {"instruction": "CI/CD", "output": "CI — автоматическая сборка. CD — автоматический деплой."},
]

print("Dataset:", len(training_data), "examples")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading", MODEL_NAME, "...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    low_cpu_mem_usage=True, trust_remote_code=True
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
    lora_dropout=0.05, bias="none", target_modules=["q_proj", "v_proj"]
)
model = get_peft_model(model, lora_config)
model = model.to(device)
gc.collect()
torch.cuda.empty_cache()

print("Memory:", round(torch.cuda.memory_allocated() / 1024**3, 1), "GB")
model.print_trainable_parameters()

def fmt(ex):
    return {"text": "### Инструкция:\n" + ex["instruction"] + "\n\n### Ответ:\n" + ex["output"]}

dataset = Dataset.from_list([fmt(d) for d in training_data])
tok = lambda ex: tokenizer(ex["text"], truncation=True, max_length=256, padding="max_length")
tokenized = dataset.map(tok, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.1)

print("Train:", len(split["train"]), "Eval:", len(split["test"]))

args = TrainingArguments(
    output_dir="./darkchat-qwen-lora", num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, weight_decay=0.01, warmup_steps=50,
    logging_steps=10, save_steps=200, fp16=True, optim="adamw_torch",
    report_to="none", eval_strategy="steps", eval_steps=50,
    save_total_limit=2, gradient_checkpointing=True, max_grad_norm=0.3,
    lr_scheduler_type="cosine"
)

trainer = Trainer(
    model=model, args=args, train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

print("Training...")
trainer.train()
print("Done!")

trainer.save_model("./darkchat-qwen-lora")
tokenizer.save_pretrained("./darkchat-qwen-lora")
print("Model saved!")

model.eval()
def gen(prompt):
    inputs = tokenizer("### Инструкция:\n" + prompt + "\n\n### Ответ:\n", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("### Ответ:\n")[-1]

print("\n=== Test ===")
print("Q: Привет!")
print("A:", gen("Привет!"))
print("Q: Hello world Python")
print("A:", gen("Напиши hello world на Python"))
print("Q: API?")
print("A:", gen("Что такое API?"))

import shutil
from google.colab import files
shutil.make_archive("darkchat-qwen-lora", "zip", "./darkchat-qwen-lora")
files.download("darkchat-qwen-lora.zip")
print("Downloaded!")